# First examples

In [ ]:
#    APM41012EP course notebook - Chapter 2 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    First examples
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "seaborn"
import numpy as np
from scipy import interpolate
import warnings
warnings.filterwarnings('ignore')

All the examples below use Newton's method through the function of the `scipy` module: [interpolate.KroghInterpolator](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.KroghInterpolator.html#scipy.interpolate.KroghInterpolator).

## Polynomial interpolation

**Example of interpolation by a polynomial of degree 8 passing through 9 points**

In [ ]:
xi = np.array([ 0.,  1., 2., 4., 5.,  7., 8.,  9., 10.])
yi = np.array([-1., -6., 1., 6., 0., -6., 2., 12.,  5.])

p = interpolate.KroghInterpolator(xi, yi)

**Plot for $x \in [-0.1;10.1]$**

In [ ]:
xmin = -0.1; xmax = 10.1
x = np.linspace(xmin, xmax, 1000)

fig = go.Figure(layout_legend=dict(orientation="h", y=1.1))
fig.add_trace(go.Scatter(x=x, y=p(x), name="p(x)"))
fig.add_trace(go.Scatter(x=xi, y=yi, mode='markers', name="interpolation pts"))
fig.show()

**Plot for $x \in [-1;11]$** 

In [ ]:
xmin = -1.; xmax = 11.
x = np.linspace(xmin, xmax, 1000)

fig = go.Figure(layout_legend=dict(orientation="h", y=1.1))
fig.add_trace(go.Scatter(x=x, y=p(x), name="p(x)"))
fig.add_trace(go.Scatter(x=xi, y=yi, mode='markers', name="interpolation pts"))
fig.show()

## Polynomial approximation 

**Example for the function $f(x) = sin(x)$ with 11 points**

In [ ]:
n = 10
ximin = 0.; ximax = 3*np.pi
xi = np.linspace(ximin, ximax, n+1)
yi = np.sin(xi)

p = interpolate.KroghInterpolator(xi, yi)

xmin = ximin-2.; xmax = ximax+2
x = np.linspace(xmin, xmax, 1000)

fig = go.Figure(layout_title=f"Interpolating polynomial of degree {n} for the function sin(x)", layout_legend=dict(orientation="h", y=1.1))
fig.add_trace(go.Scatter(x=x, y=np.sin(x), name="sin(x)"))
fig.add_trace(go.Scatter(x=x, y=p(x), name="p(x)"))
fig.add_trace(go.Scatter(x=xi, y=yi, mode='markers', name="interpolation pts"))

fig.show()

**Example for the function $f(x) = sin(x)$ with 26 points**

In [ ]:
n = 25
ximin = 0.; ximax = 3*np.pi
xi = np.linspace(ximin, ximax, n+1)
yi = np.sin(xi)

p = interpolate.KroghInterpolator(xi, yi)

xmin = ximin-2.; xmax = ximax+2
x = np.linspace(xmin, xmax, 1000)

fig = go.Figure(layout_title=f"Perturbed interpolating polynomial of degree {n} for the function sin(x)", layout_legend=dict(orientation="h", y=1.1))
fig.add_trace(go.Scatter(x=x, y=np.sin(x), name="sin(x)"))
fig.add_trace(go.Scatter(x=x, y=p(x), name="p(x)"))
fig.add_trace(go.Scatter(x=xi, y=yi, mode='markers', name="interpolation pts"))

fig.show()

**Sensitivity to perturbations**

In [ ]:
def plot_sol_pert(eps, n, xmin, xmax, xi, yi, pn, yi_pert, pn_pert):

    xmin = ximin; xmax = ximax
    x = np.linspace(xmin, xmax, 1000)

    fig = go.Figure()

    fig.add_trace(go.Scatter(x=x, y=np.sin(x), name="sin(x)"))
    fig.add_trace(go.Scatter(x=xi, y=yi, mode='markers', name="interp. pts"))
    fig.add_trace(go.Scatter(x=x, y=pn(x), name="p(x)"))
    
    for i, epsi in enumerate(eps):
        fig.add_trace(go.Scatter(visible=False, x=xi, y=yi_pert[i], mode='markers', name=f"perturbed interp. pts"))
        fig.add_trace(go.Scatter(visible=False, x=x, y=pn_pert[i](x), name=f"p(x) with eps = {epsi}"))
    
    # Make plot visible for eps=1e-5
    fig.data[9].visible = True
    fig.data[10].visible = True
    
    # Create and add slider
    steps = []
    for i, epsi in enumerate(eps):
        step = dict(method="update", label = f" {epsi}",
                    args=[{"visible": [el==0 or el==1 or el==2 or el==2*i+3 or el==2*i+4 for el in range(len(fig.data))]}])
        steps.append(step)
            
    sliders = [dict(currentvalue={"prefix": "eps = "}, steps=steps, active=3)]
    legend = dict(orientation="h", x=0.1, y=1.14, bgcolor = 'rgba(0,0,0,0)')
    title = f"Interpolating polynomial of degree {n} for the function sin(x)"
    
    fig.update_layout(sliders=sliders, legend=legend, title=title, height=500)
    
    fig.show()  

In [ ]:
eps = np.array([1e-2, 1e-3, 1e-4, 1e-5])
n = 25
ximin = 0.; ximax = 3*np.pi

xi = np.linspace(ximin, ximax, n+1)
yi = np.sin(xi)
pn = interpolate.KroghInterpolator(xi, yi)

yi_pert = []; pn_pert = []
for i, epsi in enumerate(eps):
    yi_pert.append(np.sin(xi))
    yi_pert[i][11] = yi_pert[i][11] + epsi
    pn_pert.append(interpolate.KroghInterpolator(xi, yi_pert[i]))

plot_sol_pert(eps, n, xmin, xmax, xi, yi, pn, yi_pert, pn_pert)

**Example for the function $\displaystyle f(x) = \frac{1}{1+25x^2}$ with 11 points**

In [ ]:
def f(x):
    return 1/(1+25*x*x)

In [ ]:
n = 10
ximin = -1; ximax = 1
xi = np.linspace(ximin, ximax, n+1)
yi = f(xi)

p = interpolate.KroghInterpolator(xi, yi)

x = np.linspace(ximin, ximax, 500)

fig = go.Figure(layout_yaxis_range=[-0.5,2.0], layout_title=f"Interpolating polynomial of degree {n} for f(x)", layout_legend=dict(orientation="h", y=1.1))
fig.add_trace(go.Scatter(x=x, y=f(x), name="f(x)"))
fig.add_trace(go.Scatter(x=x, y=p(x), name="p(x)"))
fig.add_trace(go.Scatter(x=xi, y=yi, mode='markers', name="interpolation pts"))
fig.show()

**Example for the function $\displaystyle f(x) = \frac{1}{1+25x^2}$ with 26 points**

In [ ]:
n = 25

ximin = -1; ximax = 1
xi = np.linspace(ximin, ximax, n+1)
yi = f(xi)

p = interpolate.KroghInterpolator(xi, yi)

x = np.linspace(ximin, ximax, 500)

fig = go.Figure(layout_yaxis_range=[-0.5,2.0], layout_title=f"Interpolating polynomial of degree {n} for f(x)", layout_legend=dict(orientation="h", y=1.1))
fig.add_trace(go.Scatter(x=x, y=f(x), name="f(x)"))
fig.add_trace(go.Scatter(x=x, y=p(x), name="p(x)"))
fig.add_trace(go.Scatter(x=xi, y=yi, mode='markers', name="interpolation pts"))
fig.show()